In [199]:
import json
import html
import openai
import re

In [245]:

PROMPT_CONTEXT = """
You are a financial report writer, you need to create financial report in SSML according to the following data:

"""

DATA_TEMPLATE = """

Security Name: APPL
Trending: 
Open: $57.30
Close: $50.30
Trending: Down 7%

Volumn: 
Total: $0.157 Billion
Compare to 5 Day Avg: 150%
Active Buying: $0.1 Billion
Active Selling: $0.057 Billion

Funding Flow: 
Day: Out $12 Million
Days Count: 3Days

Main Funds:
Percentage: 50%
Active Buying: 5%
Active Selling: 15%

"""



PAGE_POINTS = """

The SSML should at least contain the following tags corresponding to each data section above:
<mark name="Title" />
<mark name="Trending" />
<mark name="Volumn" />
<mark name="Capital Flows" />
<mark name="Main Fund" />

"""


CHART_POINTS = """

You also need to add tags to data descriptions.
<mark name="Volumn Compare To 5 Day Avg" />
<mark name="Volumn Active Selling And Buying" />
<mark name="Main Fund Buying" />
<mark name="Main Fund Selling"/>

"""


RULES = """

You need to add <say-as interpret-as="verbatim"/> tag for Security Name. 

Do not add any tags other then mentioned above.

"""


EXAMPLE = """

Here is an example:
<mark name="Title" />June 1st Stock Review for <say-as interpret-as="verbatim">CITIC</say-as>. 
<mark name="Treding" />Today, <say-as interpret-as="verbatim">CITIC</say-as> Securities experienced a slight decrease in its stock price, falling by 0.55% to close at 25.25 yuan. 
<mark name="Volumn" />The total transaction amount for the day was 389 million yuan. <mark name="Volumn Compare To 5 Day Avg" />which is higher than the five-day average at 123%. 
<mark name="Volumn Active Selling And Buying" />Among the transactions, there were 168 million yuan in active buying and 221 million yuan in active selling. 
<mark name="Capital Flows" />However, the overall trend of funds showed a net outflow throughout the day, with a net outflow of 53.5644 million yuan. This stock has seen 13 consecutive days of net outflow of funds.
<mark name="Main Fund" />In terms of main funds, the main trading ratio of <say-as interpret-as="verbatim">CITIC</say-as> Securities was 33%,
<mark name="Main Fund Buying" />with a main active buying ratio of 5%
<mark name="Main Fund Selling" />and a main active selling ratio of 15%.

You need to mimic the example above.

"""


PROMPT_START = """

Your report:
"""


In [246]:
prompt = f"{PROMPT_CONTEXT} \n {DATA_TEMPLATE} \n {PAGE_POINTS} \n {CHART_POINTS} \n {RULES} \n {EXAMPLE} \n {PROMPT_START} \n"

In [247]:
page_dict = dict([(f'"{p}"', f'"{p} Page"'.replace(" ", "-")) for p in re.findall(r'"([^"]*)"', PAGE_POINTS)])

In [183]:
chart_dict = dict([(f'"{c}"', f'"{c} Chart"'.replace(" ", "-")) for c in re.findall(r'"([^"]*)"', CHART_POINTS)])

In [184]:
all_dict = {}
all_dict.update(page_dict)
all_dict.update(chart_dict)

In [185]:
all_dict

{'"Title"': '"Title-Page"',
 '"Treding"': '"Treding-Page"',
 '"Volumn"': '"Volumn-Page"',
 '"Capital Flows"': '"Capital-Flows-Page"',
 '"Main Fund"': '"Main-Fund-Page"',
 '"Volumn Compare To 5 Day Avg"': '"Volumn-Compare-To-5-Day-Avg-Chart"',
 '"Volumn Active Selling And Buying"': '"Volumn-Active-Selling-And-Buying-Chart"',
 '"Main Fund Buying"': '"Main-Fund-Buying-Chart"',
 '"Main Fund Selling"': '"Main-Fund-Selling-Chart"'}

In [75]:

def generate_name_to_html_id_dict(template):
    result = {}
    for line in DATA_TEMPLATE.split("\n"):
        if line:
            line_split = [s for s in line.split(":") if s]
            name = line_split[0]
            if len(line_split) > 1:
                p_name = f"{name} Chart"
            else:
                p_name = f"{name} Page"
            
            html_id = p_name.replace(" ", "-")
            
            result[name] = html_id
    return result
            


In [76]:
name_to_id = generate_name_to_html_id_dict(DATA_TEMPLATE)

In [77]:
def augment_ssml(name_to_id, ssml):
    for k,v in name_to_id.items():
        ssml = ssml.replace(k,v)
        
    return ssml

In [78]:
async def completion(prompt, model="gpt-3.5-turbo"):
    response = await openai.ChatCompletion.acreate(
                model=model,
                messages=[{"role": "system", "content": prompt}],
            )
    return response.choices[0].message["content"]

def wrap_ssml(prompt):
    return f"<speak>{prompt}<mark name=\"End\"/></speak>"

In [22]:


gpt_prompt_zh = """
你是一个撰稿人，你需要根据如下数据撰写SSML播报：

基金：普源精电
涨跌行情：
今开：57.30元
收盘：50.30元
今日下跌：7%

成交额：
总成交额：1.57亿元
五日成交额比率：150%
主动买入：1亿元
主动卖出：0.57亿元

资金流向：
全天净流出：1234万元
流出天数：3天

主力资金：
主力成交：50%
主动买入：5%
主动卖出：15%

SSML播报需要包含如下几个部分：
<mark name="标题" />
<mark name="涨跌行情" />
<mark name="成交额" />
<mark name="资金流向" />
<mark name="主力资金" />

你还需要为数据的描述添加标签，
<mark name="对比过去五日" />
<mark name="成交额主动被动情况" />
<mark name="主力主动买入" />
<mark name="主力主动卖出"/>

不要添加除上面提到以外的其他任何标签。


这是一个例子：
<mark name="标题" />中信建投6月1日收盘回顾，
<mark name="涨跌行情" />中信建投今日下跌0.55%，收盘价为25.25元。
<mark name="成交额" />今日总成交额为3.89亿元，<mark name="对比过去五日" />是过去五日均值的百分之一百二十三，<mark name="成交额主动被动情况" />其中主动买入1.68亿元，主动卖出2.21亿元。
<mark name="资金流向" />全天来看，资金整体呈净流出态势，全天净流出额为5356.44万元，该股票已经连续13天资金净流出。
<mark name="主力资金" />在主力资金方面，今日中信建投的主力成交占比为33%，<mark name="主力主动" />其中主力主动买入占比5%，<mark name="主力被动"/>主力主动卖出占比15%。

你需要模仿例子的结构和字数。
播报：
"""

In [102]:
gpt_prompt_en = """
You are a financial report writer, you need to create financial report in SSML according to the following data:


Security Name: APPL
Trending: 
Open: $57.30
Close: $50.30
Trending: Down 7%

Volumn: 
Total: $0.157 Billion
Compare to 5 Day Avg: 150%
Active Buying: $0.1 Billion
Active Selling: $0.057 Billion

Funding Flow: 
Day: Out $12 Million
Days Count: 3Days

Main Funds:
Percentage: 50%
Active Buying: 5%
Active Selling: 15%

The SSML should at least contain the following tags corresponding to each data section above:
<mark name="Title" />
<mark name="Treding" />
<mark name="Volumn" />
<mark name="Capital Flows" />
<mark name="Main Fund" />

You also need to add tags to data descriptions.
<mark name="Volumn Compare To 5 Day Avg" />
<mark name="Volumn Active Selling And Buying" />
<mark name="Main Fund Buying" />
<mark name="Main Fund Selling"/>

You need to add <say-as interpret-as="verbatim"/> tag for Security Name. 

Do not add tags other then mentioned above

Here is an example:
<mark name="Title" />June 1st Stock Review for <say-as interpret-as="verbatim">CITIC</say-as>. 
<mark name="Treding" />Today, <say-as interpret-as="verbatim">CITIC</say-as> Securities experienced a slight decrease in its stock price, falling by 0.55% to close at 25.25 yuan. 
<mark name="Volumn" />The total transaction amount for the day was 389 million yuan. <mark name="Volumn Compare To 5 Day AVG" />which is higher than the five-day average at 123%. 
<mark name="Volumn Active Selling And Buying" />Among the transactions, there were 168 million yuan in active buying and 221 million yuan in active selling. 
<mark name="Capital Flows" />However, the overall trend of funds showed a net outflow throughout the day, with a net outflow of 53.5644 million yuan. This stock has seen 13 consecutive days of net outflow of funds.
<mark name="Main Fund" />In terms of main funds, the main trading ratio of <say-as interpret-as="verbatim">CITIC</say-as> Securities was 33%,
<mark name="Main Fund Buying" />with a main active buying ratio of 5%
<mark name="Main Fund Selling" />and a main active selling ratio of 15%.

You need to mimic the structure and word counts in the example above.

Your report:

"""



In [221]:
print(gpt_prompt_en)


You are a financial report writer, you need to create financial report in SSML according to the following data:


Security Name: APPL
Trending: 
Open: $57.30
Close: $50.30
Trending: Down 7%

Volumn: 
Total: $0.157 Billion
Compare to 5 Day Avg: 150%
Active Buying: $0.1 Billion
Active Selling: $0.057 Billion

Funding Flow: 
Day: Out $12 Million
Days Count: 3Days

Main Funds:
Percentage: 50%
Active Buying: 5%
Active Selling: 15%

The SSML should at least contain the following tags corresponding to each data section above:
<mark name="Title" />
<mark name="Treding" />
<mark name="Volumn" />
<mark name="Capital Flows" />
<mark name="Main Fund" />

You also need to add tags to data descriptions.
<mark name="Volumn Compare To 5 Day Avg" />
<mark name="Volumn Active Selling And Buying" />
<mark name="Main Fund Buying" />
<mark name="Main Fund Selling"/>

You need to add <say-as interpret-as="verbatim"/> tag for Security Name. 

Do not add tags other then mentioned above

Here is an example:
<

In [222]:
print(prompt)


You are a financial report writer, you need to create financial report in SSML according to the following data:

 
 

Security Name: APPL
Trending: 
Open: $57.30
Close: $50.30
Trending: Down 7%

Volumn: 
Total: $0.157 Billion
Compare to 5 Day Avg: 150%
Active Buying: $0.1 Billion
Active Selling: $0.057 Billion

Funding Flow: 
Day: Out $12 Million
Days Count: 3Days

Main Funds:
Percentage: 50%
Active Buying: 5%
Active Selling: 15%

 
 

The SSML should at least contain the following tags corresponding to each data section above:
<mark name="Title" />
<mark name="Trending" />
<mark name="Volumn" />
<mark name="Capital Flows" />
<mark name="Main Fund" />

 
 

You also need to add tags to data descriptions.
<mark name="Volumn Compare To 5 Day Avg" />
<mark name="Volumn Active Selling And Buying" />
<mark name="Main Fund Buying" />
<mark name="Main Fund Selling"/>

 
 

You need to add <say-as interpret-as="verbatim"/> tag for Security Name. 

Do not add any tags other then mentioned abov

In [248]:
gpt_response = await completion(prompt=prompt)

In [249]:
gpt_response = augment_ssml(all_dict, gpt_response)

In [250]:
gpt_response = wrap_ssml(gpt_response)


```python
gpt_result = """
<mark name="Title" />Financial Report for <say-as interpret-as="verbatim">APPL</say-as> 
<mark name="Treding" />Today, the stock price of <say-as interpret-as="verbatim">APPL</say-as> opened at $57.30 and closed at $50.30, which is down 7%. 
<mark name="Volumn" />The total transaction amount for the day was $0.157 Billion. <mark name="Compare_To_5_Day_AVG" />This is higher than the five-day average at 150%. 
<mark name="Volumn_Active_Selling_And_Buying" />Among the transactions, there were $0.1 billion in active buying and $0.057 billion in active selling. 
<mark name="Capital_Flows" />However, the overall trend of funds showed a net outflow throughout the day, with a net outflow of $12 Million. This stock has been experiencing a net outflow of funds for the past 3 days.
<mark name="Main_Fund" />In terms of main funds, the main trading ratio of <say-as interpret-as="verbatim">APPL</say-as> was 50%.
<mark name="Main_Fund_Buying" />With a main active buying ratio of 5%, 
<mark name="Main_Fund_Selling" />and a main active selling ratio of 15%.
"""

gpt_result = f"<speak>{gpt_result}<mark name=\"End\"/></speak>"
```

In [251]:
print(gpt_response)

<speak><mark name="Title-Page" />Financial Report for <say-as interpret-as="verbatim">APPL</say-as>. 
<mark name="Trending" />Today, <say-as interpret-as="verbatim">APPL</say-as> experienced a downtrend in its stock price, opening at $57.30 and closing at $50.30, which is down 7%. 
<mark name="Volumn-Page" />The total transaction amount for the day was $0.157 Billion. <mark name="Volumn-Compare-To-5-Day-Avg-Chart" />This is higher than the five-day average at 150%. 
<mark name="Volumn-Active-Selling-And-Buying-Chart" />Among the transactions, there were active buying and selling of $0.1 Billion and $0.057 Billion, respectively. 
<mark name="Capital-Flows-Page" />In capital flows, there was a net outflow of $12 Million today and this trend has been going on for 3 days. 
<mark name="Main-Fund-Page" />In terms of main funds, the main trading ratio of <say-as interpret-as="verbatim">APPL</say-as> was 50%,
<mark name="Main-Fund-Buying-Chart" />with a main active buying ratio of 5%
<mark nam

In [31]:
import os
from google.cloud import texttospeech_v1beta1 as texttospeech

In [32]:
def timepoints_to_json(resp):
    return list({"mark_name":ele.mark_name, "time_seconds": ele.time_seconds} for ele in resp)

In [33]:

# [START tts_ssml_address_audio]
async def ssml_to_audio(ssml_text, lang_code, outfile):
    # Generates SSML text from plaintext.
    #
    # Given a string of SSML text and an output file name, this function
    # calls the Text-to-Speech API. The API returns a synthetic audio
    # version of the text, formatted according to the SSML commands. This
    # function saves the synthetic audio to the designated output file.
    #
    # Args:
    # ssml_text: string of SSML text
    # outfile: string name of file under which to save audio output
    #
    # Returns:
    # nothing

    # Instantiates a client
    client = texttospeech.TextToSpeechAsyncClient()

    # Sets the text input to be synthesized
    synthesis_input = texttospeech.SynthesisInput(ssml=ssml_text)

    # Builds the voice request, selects the language code ("en-US") and
    # the SSML voice gender ("MALE")
    voice = texttospeech.VoiceSelectionParams(
        language_code=lang_code, ssml_gender=texttospeech.SsmlVoiceGender.MALE
    )

    # Selects the type of audio file to return
    audio_config = texttospeech.AudioConfig(
        audio_encoding=texttospeech.AudioEncoding.MP3
    )
    
    request = texttospeech.SynthesizeSpeechRequest(
        input=synthesis_input,
        voice=voice,
        audio_config=audio_config,
        enable_time_pointing=[texttospeech.SynthesizeSpeechRequest.TimepointType(1)]
    )

    # Performs the text-to-speech request on the text input with the selected
    # voice parameters and audio file type
    response = await client.synthesize_speech(request=request)
    
    list_of_json = timepoints_to_json(response.timepoints)
    
    print(response.timepoints)
    # Writes the synthetic audio to the output file.
    with open(outfile, "wb") as out:
        out.write(response.audio_content)
        print("Audio content written to file " + outfile)

    with open(f"{outfile}.json", "w", encoding="utf8") as out:
        json.dump(list_of_json, indent=4, fp=out)
        
    return 
    # [END tts_ssml_address_audio]


In [252]:
await ssml_to_audio(gpt_response, "en_US", "out.mp3")

[mark_name: "Title-Page"
time_seconds: 0.0099999997764825821
, mark_name: "Trending"
time_seconds: 2.420875072479248
, mark_name: "Volumn-Page"
time_seconds: 13.120753288269043
, mark_name: "Volumn-Compare-To-5-Day-Avg-Chart"
time_seconds: 18.539628982543945
, mark_name: "Volumn-Active-Selling-And-Buying-Chart"
time_seconds: 22.556753158569336
, mark_name: "Capital-Flows-Page"
time_seconds: 31.529294967651367
, mark_name: "Main-Fund-Page"
time_seconds: 38.456211090087891
, mark_name: "Main-Fund-Buying-Chart"
time_seconds: 43.435420989990234
, mark_name: "Main-Fund-Selling-Chart"
time_seconds: 45.976131439208984
, mark_name: "End"
time_seconds: 49.293333333333337
]
Audio content written to file out.mp3


In [253]:
import IPython
IPython.display.Audio("out.mp3")


In [ ]:
json.load()